# ASIC1a pI-code MD, part 3: compare the three builds

Taner Karagol and Alper Karagol.

**Runtime > Run all.** No GPU needed and nothing to fill in but the project folder.
It takes under a minute.

Open this once the three per-build notebooks have each finished and written their
report. It reads the three JSON files off Drive, so it does not matter that the
builds ran in three separate sessions.

It refuses to compare builds that are not comparable: different sampling, different
box sizes, or a pocket pair missing from one of them.

**What it writes to Drive**

    results/comparison.json     the three-way comparison and the verdict
    results/md_comparison.png   the figure
    results/SUMMARY.txt         the whole thing in plain text


In [ ]:
#@title 1. Settings and the three reports
PROJECT = "/content/drive/MyDrive/MDs/ASIC1a_picode"  #@param {type:"string"}
SUFFIX  = ""  #@param ["", "_short"]
#@markdown Empty compares the full runs; `_short` compares part 2b's
#@markdown equal-length cuts.

from google.colab import drive
drive.mount('/content/drive')

import os, json, datetime, time
T_START = time.time()
BUILD_ORDER = ["wt", "ladder", "compare"]
RESULTS = os.path.join(PROJECT, "results")
INPUTS = os.path.join(PROJECT, "inputs")

SPEC_PATH = os.path.join(INPUTS, "build_spec.json")
if not os.path.isfile(SPEC_PATH):
    raise SystemExit("STOP: no %s. Run part 1 first." % SPEC_PATH)
SPEC = json.load(open(SPEC_PATH))
POSITION = SPEC["position"]
EXPECT = SPEC["expect"]

waiting = [b for b in BUILD_ORDER
           if not os.path.isfile(os.path.join(RESULTS, "%s_report%s.json" % (b, SUFFIX)))]
if waiting:
    raise SystemExit("STOP: still waiting on %s. Each build has its own notebook: "
                     "ASIC1a_picode_MD_2_<build>.ipynb. Run those first." % waiting)


def progress(**kw):
    p = os.path.join(RESULTS, "progress_compare.json")
    cur = json.load(open(p)) if os.path.isfile(p) else {}
    cur.update(kw)
    cur["updated"] = datetime.datetime.now().isoformat(timespec="seconds")
    json.dump(cur, open(p, "w"), indent=2)
    return cur


print("position %d, %s" % (POSITION, EXPECT))
for b in BUILD_ORDER:
    r = json.load(open(os.path.join(RESULTS, "%s_report%s.json" % (b, SUFFIX))))
    print("  %-8s %5.1f ns, %d frames, RMSD %.2f A, written %s"
          % (b, r["ns"], r["frames"], r["rmsd_mean"], r["written"]))

In [ ]:
#@title 2. The comparison, the figure and the summary
import json, os, math
import numpy as np
from scipy import stats

REPORTS = {}
for b in BUILD_ORDER:
    p = os.path.join(RESULTS, "%s_report%s.json" % (b, SUFFIX))
    if not os.path.isfile(p):
        raise SystemExit("STOP: missing %s. The previous cell did not finish %s."
                         % (p, b))
    REPORTS[b] = json.load(open(p))

ns = {b: r["ns"] for b, r in REPORTS.items()}
if max(ns.values()) - min(ns.values()) > 0.2 * max(ns.values()):
    raise SystemExit("STOP: the three builds differ by more than a fifth in sampling "
                     "(%s), so the distances are not comparable. Press Run all again "
                     "to finish the short one." % ns)
atoms = {b: r["atoms"] for b, r in REPORTS.items()}
if max(atoms.values()) - min(atoms.values()) > 3000:
    raise SystemExit("STOP: the builds differ by %d atoms (%s). Different box sizes "
                     "make a pocket distance incomparable; rebuild them in CHARMM-GUI "
                     "with the same water box and ion settings."
                     % (max(atoms.values()) - min(atoms.values()), atoms))

# Only the pocket pairs are tested; the position-246 distance has a different
# tip atom per build, so it is reported only.
labs = sorted(k for k in REPORTS["wt"]["distances"] if "|%d" % POSITION not in k)
for b in BUILD_ORDER:
    gone = [l for l in labs if l not in REPORTS[b]["distances"]]
    if gone:
        raise SystemExit("STOP: build %s does not carry %s" % (b, gone))

dl, dc = [], []
lines = ["ASIC1a pI-code MD, position %d" % POSITION,
         "%s on the ladder against %s off it, both from %s"
         % (EXPECT["ladder"], EXPECT["compare"], EXPECT["wt"]),
         "",
         "%-26s %8s %8s %8s   %9s %9s"
         % ("pocket pair", "wt", "ladder", "compare", "|lad-wt|", "|cmp-wt|")]
for l in labs:
    w = REPORTS["wt"]["distances"][l]["mean"]
    a = REPORTS["ladder"]["distances"][l]["mean"]
    c = REPORTS["compare"]["distances"][l]["mean"]
    dl.append(abs(a - w)); dc.append(abs(c - w))
    lines.append("%-26s %8.2f %8.2f %8.2f   %9.2f %9.2f"
                 % (l, w, a, c, abs(a - w), abs(c - w)))
dl, dc = np.array(dl), np.array(dc)

k = int((dl < dc).sum())
n = len(dl)
sign_p = min(1.0, 2 * sum(math.comb(n, i) for i in range(min(k, n - k) + 1)) / 2.0 ** n)
W = stats.wilcoxon(dl, dc) if n >= 6 else None
significant = (sign_p < 0.05) or (W is not None and W[1] < 0.05)
noncontact = [l for l in labs
              if all(REPORTS[b]["distances"][l]["closed_fraction"] == 0.0
                     for b in BUILD_ORDER)]
verdict = ("the two substitutions differ in pocket geometry"
           if significant
           else "neither substitution moves the pocket from wild type beyond the "
                "subunit-to-subunit scatter, and the two are indistinguishable from "
                "each other")

lines += ["",
          "mean departure from wild type over %d pocket pair measurements" % n,
          "   ladder   %.3f A" % dl.mean(),
          "   compare  %.3f A" % dc.mean(),
          "",
          "ladder closer to wild type in %d of %d, sign test p = %.3f"
          % (k, n, sign_p)]
if W is not None:
    lines.append("Wilcoxon signed-rank p = %.3f" % W[1])

reach = {}
for b in BUILD_ORDER:
    r = REPORTS[b]
    kk = [x for x in r["distances"] if "|%d" % POSITION in x]
    reach[b] = float(np.mean([r["distances"][x]["mean"] for x in kk]))
lines += ["",
          "position %d reach across the interface, mean over 3 subunits" % POSITION,
          "   " + ",  ".join("%s %s %.2f A" % (b, REPORTS[b]["tip_atom"], reach[b])
                             for b in BUILD_ORDER),
          "",
          "backbone RMSD, A: " + ", ".join("%s %.2f" % (b, REPORTS[b]["rmsd_mean"])
                                           for b in BUILD_ORDER),
          "sampling, ns:     " + ", ".join("%s %.1f" % (b, ns[b]) for b in BUILD_ORDER),
          "atoms:            " + ", ".join("%s %d" % (b, atoms[b])
                                           for b in BUILD_ORDER),
          "",
          "never within 4 A in any build (not a contact here): "
          + (", ".join(noncontact) if noncontact else "none"),
          "",
          "%s (sign p %.2f, Wilcoxon p %.2f). The mean departures differ by %.3f A, "
          "carried by the D237-D351 pairs, which barely form contacts."
          % (verdict, sign_p, W[1] if W is not None else float("nan"),
             abs(dl.mean() - dc.mean())),
          "The engineered side chain at %d does not reach the neighbouring D409 in any "
          "build (%.1f to %.1f A). This run is a structural control over one %d ns "
          "trajectory per build with the three subunits as the only replication; it "
          "tests whether either substitution remodels the pocket, not what either does "
          "to pH50." % (POSITION, min(reach.values()), max(reach.values()),
                        int(round(max(ns.values()))))]

SUMMARY = "\n".join(lines)
print(SUMMARY)

json.dump(dict(position=POSITION, expect=EXPECT, labels=labs,
               ladder_departure=list(map(float, dl)),
               compare_departure=list(map(float, dc)),
               ladder_mean=float(dl.mean()), compare_mean=float(dc.mean()),
               ladder_closer_in=k, n=n, sign_p=float(sign_p),
               wilcoxon_p=float(W[1]) if W is not None else None,
               reach=reach, ns=ns, atoms=atoms, verdict=verdict,
               significant=bool(significant), noncontact_pairs=noncontact),
          open(os.path.join(RESULTS, "comparison%s.json" % SUFFIX), "w"), indent=2)
open(os.path.join(RESULTS, "SUMMARY%s.txt" % SUFFIX), "w").write(SUMMARY + "\n")

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(9.2, 3.8))
ax = axes[0]
x = np.arange(n)
ax.bar(x - 0.2, dl, 0.38, color="#2166ac", edgecolor="#1a1a1a", lw=0.5,
       label="pI-code, %d%s" % (POSITION, REPORTS["ladder"]["tip_atom"]))
ax.bar(x + 0.2, dc, 0.38, color="#b2182b", edgecolor="#1a1a1a", lw=0.5,
       label="comparator, %d%s" % (POSITION, REPORTS["compare"]["tip_atom"]))
ax.set_xticks(x)
ax.set_xticklabels([l.split("|")[1] + "\n" + l.split("|")[0] for l in labs], fontsize=6)
ax.set_ylabel("departure from wild type (A)")
ax.set_title("a   pocket pair distances", fontsize=8, loc="left")
ax.legend(fontsize=6.5, frameon=False)
ax = axes[1]
ax.axline((0, 0), slope=1, color="#9a9a9a", lw=0.8, ls="--")
ax.scatter(dc, dl, s=30, facecolor="#2166ac", edgecolor="#1a1a1a", lw=0.4)
m = max(dl.max(), dc.max()) * 1.15
ax.set_xlim(0, m); ax.set_ylim(0, m); ax.set_aspect("equal")
ax.set_xlabel("comparator departure (A)")
ax.set_ylabel("pI-code departure (A)")
ax.set_title("b   departure from wild type, both substitutions", fontsize=8,
             loc="left")
ax.text(0.04 * m, 0.93 * m, "scatter about the diagonal: no systematic difference",
        fontsize=6, va="top", color="#555555")
fig.tight_layout()
fig.savefig(os.path.join(RESULTS, "md_comparison%s.png" % SUFFIX), dpi=300)
progress(finished=True, verdict=verdict)
print("\nwrote SUMMARY.txt, comparison.json and md_comparison.png to %s" % RESULTS)
fig

## Reading it

Everything is in `results/SUMMARY.txt` on Drive, so there is nothing to scroll back for.

**If the ladder build departs further from wild type than the comparator does**, the
structural result agrees with the functional one: at the most buried position in the
pocket the near-isosteric substitution is the more disruptive one, and isostericity
bought nothing. That is what this project expects, and finding it here makes the
negative stronger instead of merely repeating it.

**If the ladder build sits closer**, the structural premise survives at this position
while the functional consequence does not follow from it. That is a different and more
interesting paper: the ladder would be conserving local structure, and conserved local
structure would be failing to predict the proton-sensing phenotype. Report it.

**What would make either reading unsafe.** Cell 5 refuses outright on the last two and
prints a warning on the first:

- Backbone RMSD much above 6 angstrom in any build, which usually means a disulfide was
  not declared in CHARMM-GUI.
- Atom counts differing by more than 3000 between builds, which means the boxes differ.
- Sampling differing by more than a fifth between builds.

Twenty nanoseconds per build settles a side chain and a local hydrogen-bond network. It
does not sample a gating transition, and no claim about gating should come out of it.

**One position is one position.** To run the opposite case, go back to part 1, set
POSITION 418, WT_RESNAME GLU, LADDER_TO GLN, COMPARE_TO ALA and a different PROJECT
folder, then come back here with that same PROJECT. At E418 the ladder substitution
E418Q wins by 0.350 pH units against E418A, the largest ladder win in the measured set.
Two positions where the measurement points in opposite directions is a far better test
than either alone.